<a href="https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZuhaaAsif/Applied-Search-Intelligence-Google-Search-Ranking-Discoverability/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
# setup
%pip -q install duckdb huggingface_hub
import os, getpass, json
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

metrics = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions, SUM(gsc_clicks) AS clicks,
           SUM(CASE WHEN gsc_sum_position >= gsc_impressions THEN gsc_sum_position ELSE 0 END) AS sum_position,
           SUM(CASE WHEN gsc_sum_position >= gsc_impressions THEN gsc_impressions ELSE 0 END) AS impressions_with_position
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2 HAVING SUM(gsc_impressions) > 0
""").df()
metrics['avg_position'] = metrics['sum_position'] / metrics['impressions_with_position']
metrics['ctr'] = metrics['clicks'] / metrics['impressions']

content_meta = con.sql(f"SELECT content_hash_id, content_type, content_created_date FROM read_parquet('{REL}/dim_content.parquet')").df()
metrics = metrics.merge(content_meta, on='content_hash_id', how='left')
metrics['content_age_days'] = (pd.Timestamp('2026-03-31') - pd.to_datetime(metrics['content_created_date'])).dt.days

signal_df = metrics[(metrics['avg_position'] > 0) & (metrics['impressions'] >= 100)].copy()
signal_df['position_bin'] = pd.cut(signal_df['avg_position'], bins=[0,3,10,20,50,100000], labels=['1-3','4-10','11-20','21-50','51+'])
tier_ctr = signal_df.groupby('position_bin', observed=True)['ctr'].mean()
signal_df['tier_expected_ctr'] = signal_df['position_bin'].map(tier_ctr).astype(float)
signal_df['ctr_gap'] = signal_df['tier_expected_ctr'] - signal_df['ctr']

underperforming = (signal_df['ctr_gap'] > 0).astype(int)
signal_df['score'] = underperforming * signal_df['ctr_gap'] * signal_df['impressions']
print(f"{len(signal_df):,} pages with real position + volume")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,424 pages with real position + volume


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Three reason codes, each mapped to a distinct action:

1. high_rank_low_ctr (24,041 pages, 35.0% of the flagged queue): good position, near-zero clicks, likely a title/snippet problem; action: review_title_meta.
2. aging_and_ctr_gap (16,847 pages, 24.5%): 270+ days old and gapping, decay compounding with a CTR problem; action: refresh_and_review_metadata. This reflects the paper's own Finding 2/4 decay pattern (health peaking around 61–90 days, declining past 270), applied here to my own validated CTR-gap signal rather than the paper's health score.
3. ctr_below_tier_expectation (27,764 pages, 40.4%): the general case. Age is checked before gap size, so a page that's both old and gapping is coded as the decay case, not the metadata case, even when it would also qualify for both, the more specific, more actionable explanation wins. Of 101,424 scored pages, 68,652 (67.7%) cleared the gap threshold and made the queue; ranking within each code uses the same transparent score as ML-07 (ctr_gap × impressions). Several top-ranked pages here are the same content items that topped the ML-07 queue weeks earlier, the ranking is stable and reproducible across two independent builds of this pipeline, not a one-off artifact.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def assign_reason(row):
    if row['ctr_gap'] <= 0:
        return 'not_underperforming'
    if row['content_age_days'] >= 270 and row['ctr_gap'] > 0:
        return 'aging_and_ctr_gap'          # decay + CTR gap together — strongest refresh case
    if row['position_bin'] in ['1-3', '4-10'] and row['ctr_gap'] > 0.001:
        return 'high_rank_low_ctr'          # near-zero CTR at a good position — likely metadata issue
    return 'ctr_below_tier_expectation'     # general case

signal_df['reason_code'] = signal_df.apply(assign_reason, axis=1)

action_map = {
    'high_rank_low_ctr': 'review_title_meta',
    'aging_and_ctr_gap': 'refresh_and_review_metadata',
    'ctr_below_tier_expectation': 'review_title_meta',
    'not_underperforming': 'monitor',
}
signal_df['action'] = signal_df['reason_code'].map(action_map)

queue = signal_df[signal_df['ctr_gap'] > 0].sort_values('score', ascending=False)
print(queue['reason_code'].value_counts())
queue[['content_hash_id','position_bin','ctr','tier_expected_ctr','content_age_days','score','reason_code','action']].head(20)

reason_code
ctr_below_tier_expectation    27764
high_rank_low_ctr             24041
aging_and_ctr_gap             16847
Name: count, dtype: int64


,content_hash_id,position_bin,ctr,tier_expected_ctr,content_age_days,score,reason_code,action
100154,content_44f34c0a90047651,4-10,0.000113,0.003177,69,650.892748,high_rank_low_ctr,review_title_meta
32496,content_8d7d99f109e19aa2,1-3,0.001420,0.003928,375,510.378849,aging_and_ctr_gap,refresh_and_review_metadata
77681,content_8e1334d6356668e3,4-10,0.000007,0.003177,410,427.898338,aging_and_ctr_gap,refresh_and_review_metadata
125639,content_34a70fea29d15f24,4-10,0.000301,0.003177,278,411.428758,aging_and_ctr_gap,refresh_and_review_metadata
166041,content_fec55986a1868d62,4-10,0.000008,0.003177,410,393.236068,aging_and_ctr_gap,refresh_and_review_metadata
130444,content_7c6373141eae744a,4-10,0.000626,0.003177,105,338.301172,high_rank_low_ctr,review_title_meta
17926,content_f6116743b00afc2d,4-10,0.000139,0.003177,193,326.837543,high_rank_low_ctr,review_title_meta
121015,content_306bc78dff1eb683,1-3,0.000433,0.003928,375,282.481820,aging_and_ctr_gap,refresh_and_review_metadata
130474,content_acbcc847f8996314,4-10,0.001534,0.003177,105,280.725563,high_rank_low_ctr,review_title_meta
1890,content_cd3d932d4e1c8db0,4-10,0.000045,0.003177,447,279.843614,aging_and_ctr_gap,refresh_and_review_metadata


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use: a weekly-reviewed shortlist for a content reviewer with limited time, ranking which pages to check first for title/metadata or refresh review.

Where it stops being valid:
1. This queue is built from one month (March 2026) of one client cohort, not validated across other months or seasons.
2. The position calculation excludes rows where gsc_sum_position < gsc_impressions.
3. The ML-08/09 model showed the pattern behind this ranking is directionally real (precision clearing base rate by 20+ points) but only within-month, no evidence yet that March's tier-CTR relationship holds in other months.
4. content_type carried almost no predictive weight in the validated model (<0.001 importance per category), so this queue should not be read as saying certain content types are inherently worse.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A reviewer must check, before acting on any flagged page: whether a SERP feature is capturing clicks regardless of metadata quality; whether the page is a branded/navigational query where low CTR is expected, not a problem; whether a sibling page on the same site absorbed the traffic; and whether the flagged CTR is part of the ~28% zero-CTR block found in ML-07, which may reflect a measurement gap rather than a real content issue.

What should NOT be automated: auto-rewriting titles/metadata without human review; the score identifies candidates, not verified problems, and several ML-07 top-20 rows turned out to share a suspicious near-identical near-zero CTR signature that looked more like a shared tracking issue than five real content problems; any action based on a single month's data without re-confirming the pattern holds in the current month; merging or pruning pages based on this queue alone (that requires the consolidation check above, which this pipeline doesn't perform); treating content_type differences as a rule, given its near-zero validated importance.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Re-check triggers:
1. Re-run the position-validity check (gsc_sum_position >= gsc_impressions) each month, if the share of impossible rows changes meaningfully from 2.7%, the underlying data pipeline may have changed and the fix needs revisiting.
2. Re-run the grouped-split precision@K comparison from ML-09 each month the queue is used, if precision drops well below the 0.685–0.701 base-rate-beating range seen here, the tier-CTR pattern may no longer hold.
3. Watch the zero-CTR share (27.6% in March), a sharp change signals a possible tracking/measurement shift, not a genuine content trend.
4. Re-validate against a new month before trusting this queue past the month it was built on, since nothing here has been tested across months yet.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)

receipts = {
    'month': '2026-03',
    'total_pages_scored': int(len(signal_df)),
    'pages_flagged': int(len(queue)),
    'reason_code_counts': queue['reason_code'].value_counts().to_dict(),
    'position_bug_rows_affected_pct': 2.7,
    'position_bug_items_affected_pct': 38.7,
    'zero_ctr_share_pct': 27.6,
    'ml09_grouped_precision_at_20': 0.95,
    'ml09_grouped_precision_at_50': 0.90,
    'ml09_grouped_base_rate': 0.685,
    'ml09_test_clients': 10,
}
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(receipts, f, indent=2)

print(f"Wrote {len(queue):,} rows to work/outputs/action_playbook_queue.csv")
print("Wrote work/outputs/playbook_metrics.json")

Wrote 68,652 rows to work/outputs/action_playbook_queue.csv
Wrote work/outputs/playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.